# 有限域计算实验

本附录通过可执行的 Python 程序探索有限域运算、本原元、共轭、迹、基和插值。程序使用系数位串编码存储二元域元素；编码与普通整数运算不同。

使用 Python 3 内核，从上往下执行。修改输入后重新运行，比较参数变化带来的结果。断言用于核验有限例子，不替代一般定理的证明。

In [1]:
from finitefield import GF2m, irreducible
F4 = GF2m(0b111)
F16 = GF2m(0b10011)
F256 = GF2m(0b100011011)
assert not irreducible(0b101)  # x²+1 在 F2 中可约
print("F4 的 2×2（编码）：", F4.mul(2, 2))

F4 的 2×2（编码）： 3


## 1. GF(16) 的指数表与算例

模多项式为 $x^4+x+1$，生成元 $\alpha=x+(f)$ 的编码为 2。指数表第 0 项为 1，指数按模 15 约化。

`exp[k]` 存储 $\alpha^k$ 的编码，`log[a]` 返回非零编码元素的指数。负的指数差也必须模 15 约化。

In [2]:
exp, log = F16.log_tables(2)
expected = [1,2,4,8,3,6,12,11,5,10,7,14,15,13,9]
assert exp == expected
print("i   x^i 的整数编码")
for i, a in enumerate(exp):
    print(f"{i:2}  {a:2}")
assert F16.add(11, 7) == 12
assert F16.mul(3, 7) == F16.mul_log(3, 7) == 9
assert F16.div(13, 10) == F16.div_log(13, 10) == 3
print("11+7 =", F16.add(11,7))
print("3×7 =", F16.mul(3,7))
print("13÷10 =", F16.div(13,10))
print("逆指数示例 3÷13 =", F16.div_log(3,13))

i   x^i 的整数编码
 0   1
 1   2
 2   4
 3   8
 4   3
 5   6
 6  12
 7  11
 8   5
 9  10
10   7
11  14
12  15
13  13
14   9
11+7 = 12
3×7 = 9
13÷10 = 3
逆指数示例 3÷13 = 12


## 2. 零、逆元与分配律

对 GF(16) 穷举域运算性质，对 GF(256) 核验全部非零元素的逆。GF(256) 的所选模多项式不可约，但编码 2 的阶为 51，故它不能直接用于生成全体非零元素的对数表。

In [3]:
for a in range(16):
    assert F16.add(a,a)==0
    for b in range(16):
        assert F16.mul(a,b)==F16.mul(b,a)
        assert F16.mul(a,b)==F16.mul_log(a,b)
        for c in range(16):
            assert F16.mul(F16.mul(a,b),c)==F16.mul(a,F16.mul(b,c))
            assert F16.mul(a,F16.add(b,c))==F16.add(F16.mul(a,b),F16.mul(a,c))
for F in [F4,F16,F256]:
    for a in range(1,F.q):
        assert F.mul(a,F.inv(a))==1
try:
    F16.div(1,0)
except ZeroDivisionError as e:
    print("除零已阻止：", e)
assert F256.order(2)==51
print("GF(16) 穷举检查通过；GF(256) 全部逆元检查通过。")

除零已阻止： 0 没有乘法逆元
GF(16) 穷举检查通过；GF(256) 全部逆元检查通过。


## 3. F₉ 的两种表示

用二元组 $(a_0,a_1)$ 表示 $a_0+a_1\alpha$，其中 $\alpha^2=2$。系数运算在模 3 下进行。比较幂表示与多项式坐标，并观察定义元和本原元的区别。

In [4]:
def add9(a,b):
    return ((a[0]+b[0])%3,(a[1]+b[1])%3)
def mul9(a,b):
    return ((a[0]*b[0]+2*a[1]*b[1])%3,(a[0]*b[1]+a[1]*b[0])%3)
def pow9(a,n):
    r=(1,0)
    for _ in range(n): r=mul9(r,a)
    return r
alpha=(0,1); zeta=(1,1)
assert pow9(alpha,4)==(1,0) and pow9(alpha,2)!=(1,0)
assert pow9(zeta,8)==(1,0) and pow9(zeta,4)!=(1,0)
assert add9(add9(pow9(zeta,2),zeta),(2,0))==(0,0)
for i in range(1,9): print(i,pow9(zeta,i))
assert [pow9(zeta,i) for i in range(1,9)]==[(1,1),(0,2),(1,2),(2,0),(2,2),(0,1),(2,1),(1,0)]

1 (1, 1)
2 (0, 2)
3 (1, 2)
4 (2, 0)
5 (2, 2)
6 (0, 1)
7 (2, 1)
8 (1, 0)


## 4. 本原元、共轭轨道和迹

程序中的 `a` 是系数位串编码。改用 `F.power(2, k)` 选取 $\alpha^k$，或令 `a = 0` 选取零元素，比较乘法阶和轨道长度。轨道长度决定在 $\mathbb F_2$ 上的次数，乘法阶决定是否为本原元。

In [5]:
a=3
orbit=[];t=a
while t not in orbit:
    orbit.append(t);t=F16.mul(t,t)
print("Frobenius 轨道：",orbit)
print("乘法阶：",F16.order(a),"极小多项式次数：",len(orbit))
primitive=[x for x in range(1,16) if F16.order(x)==15]
print("本原元编码：",primitive)
assert len(primitive)==8  # φ(15)=8
traces=[F16.trace(x) for x in range(16)]
assert set(traces)=={0,1} and traces.count(0)==8
assert F16.trace(1)==0  # Tr(1)=4=0，并不代表迹映射为零
for x in range(16):
    for y in range(16):
        assert F16.trace(x^y)==(F16.trace(x)^F16.trace(y))
print("迹为 0 的元素：",[x for x in range(16) if F16.trace(x)==0])
assert {F16.mul(x,x)^x for x in range(16)}=={x for x in range(16) if F16.trace(x)==0}

Frobenius 轨道： [3, 5, 2, 4]
乘法阶： 15 极小多项式次数： 4
本原元编码： [2, 3, 4, 5, 9, 11, 13, 14]
迹为 0 的元素： [0, 1, 2, 3, 4, 5, 6, 7]


## 5. 幂基、对偶基与正规基

对偶基通过求解 $\operatorname{Tr}(\alpha_i\beta_j)=\delta_{ij}$ 构造。小域中可以穷举 β。正规元的 Frobenius 共轭必须线性无关；仅要求共轭互不相同还不够。

In [6]:
basis=[1,2,4,8]
dual=[]
for j in range(4):
    candidates=[b for b in range(16) if all(F16.trace(F16.mul(a,b))==int(i==j) for i,a in enumerate(basis))]
    assert len(candidates)==1
    dual.append(candidates[0])
print("幂基：",basis,"对偶基：",dual)
for x in range(16):
    coords=[F16.trace(F16.mul(b,x)) for b in dual]
    reconstructed=0
    for a,c in zip(basis,coords):
        if c: reconstructed^=a
    assert reconstructed==x

def rank2(vectors):
    pivots={}
    for x in vectors:
        while x:
            p=x.bit_length()-1
            if p in pivots: x^=pivots[p]
            else: pivots[p]=x;break
    return len(pivots)
normal=[]
for x in range(16):
    orbit=[F16.power(x,2**i) for i in range(4)]
    if rank2(orbit)==4: normal.append(x)
print("正规元编码：",normal)
print("既本原又正规的编码：",sorted(set(normal)&set(primitive)))
assert normal
assert any(x not in normal for x in primitive)

幂基： [1, 2, 4, 8] 对偶基： [9, 4, 2, 1]
正规元编码： [8, 9, 10, 11, 12, 13, 14, 15]
既本原又正规的编码： [9, 11, 13, 14]


## 6.  $\mathbb F_7$ 上的 Lagrange 插值

直接实现插值公式。相同节点会使分母为零，必须拒绝；本例只使用素域，不使用整数除法或浮点数。

In [7]:
def interpolate_value(nodes,values,t,p=7):
    if len(nodes)!=len(values) or len(set(x%p for x in nodes))!=len(nodes):
        raise ValueError('节点必须模 p 互异，且节点与值数目相等')
    result=0
    for i,ai in enumerate(nodes):
        num=den=1
        for j,aj in enumerate(nodes):
            if j!=i:
                num=num*(t-aj)%p;den=den*(ai-aj)%p
        result=(result+values[i]*num*pow(den,-1,p))%p
    return result
nodes=[0,1,2];values=[1,4,2]  # f(x)=x²+2x+1
for t in range(7):
    result=interpolate_value(nodes,values,t)
    assert result==(t*t+2*t+1)%7
    print(t,result)

0 1
1 4
2 2
3 2
4 4
5 1
6 0


## 7. 非零多项式与零函数

形式多项式 $x^2-x$ 非零，但在 $\mathbb F_2$ 上取值全为 0。加入非零多线性条件后，$x_1x_2$ 在三维立方体上恰有 $2^{3-2}=2$ 个非零点，达到非零点数下界。

In [8]:
from itertools import product
assert all((x*x-x)%2==0 for x in range(2))
points=list(product(range(2),repeat=3))
nonzero=[p for p in points if p[0]*p[1]%2]
assert len(nonzero)==2**(3-2)
print("x₁x₂ 在 F₂³ 上非零的点：",nonzero)
# Schwartz–Zippel 示例：f(x,y)=x+y，S=F₇，总次数 d=1。
zeros=sum((x+y)%7==0 for x,y in product(range(7),repeat=2))
assert zeros==7
print("x+y 在 F₇² 的零点数：",zeros,"概率：",zeros/49,"上界：",1/7)

x₁x₂ 在 F₂³ 上非零的点： [(1, 1, 0), (1, 1, 1)]
x+y 在 F₇² 的零点数： 7 概率： 0.14285714285714285 上界： 0.14285714285714285


## 继续探索

修改模多项式、节点、基和实验元素，观察哪些结论依赖这些选择。先将计算结果写成猜想，再回到正文中的定理核对条件并给出一般证明。